# Day 8b — does the length effect survive a cross-layer transcoder?

> **Supersedes `day8_clt_length_effect.ipynb`**, which had a feasibility gate that hard-stopped on a
> VRAM *estimate*, an arms table wrongly claiming `llama-plt` fits a free T4 (it is 17.3 GB), and —
> after a bad patch — markdown in code cells. If a notebook prints `0 enc, 0 dec files`, it is the
> old one.

**One question.** Replacement score rose with frame length in 27 of 27 matched short/long pairs on
GemmaScope per-layer transcoders. Does it still rise with a **cross-layer transcoder**?

Nothing else is re-run. No re-rating, no new corpus, no confirmatory tests. The 40 human ratings are
per-layer-specific and P5 is unchanged.

## Pre-registered predictions, before any transcoder is loaded

| | Prediction | Threshold |
|---|---|---|
| **C0** | The length effect replicates on a second model with per-layer transcoders | ≥ 24 of the matched pairs positive |
| **C1** | It survives the cross-layer transcoder | ≥ 24 of the matched pairs positive |
| **C2** | The per-token slope is smaller under the CLT **on the same model** | CLT slope < PLT slope |
| **C3** | Absolute replacement is higher under the CLT | CLT mean > PLT mean, same prompts |
| **C4** | Mean path length is shorter under the CLT | CLT mean < PLT mean, same prompts |

C2 is the bet the feature-count analysis implies: if about half the Gemma slope was feature count, and
the CLT changes how features are organised, the slope should move. C3 and C4 check the CLT behaves as
the literature reports (published 0.61 vs 0.37 replacement, 2.3 vs 3.7 hops) — if they fail, suspect
the setup before the finding.

**If C0 fails**, the length effect is Gemma-specific and C1 is uninterpretable. **If C1 fails**, the
effect is dictionary-dependent and the write-up's claim that it "follows from the residual stream and
should be dictionary-independent" is withdrawn. **If C1 holds and C2 fails**, the feature-count
reading weakens and the residual-stream reading strengthens.

## One caveat this design introduces

The prompts were built and validated for Gemma-2-2B — the `Fact:` prefix, the answer rates, the
category checks. Another model may not answer them, and an attribution graph explains whatever the
model actually predicted. Part 3 measures the answer rate before any comparison. A low rate does not
invalidate the length test (the metric's response to padding is still that) but it changes what the
graphs are about, and must be reported.

## Part 0 — Install

Run the three cells below, then **restart the kernel** (Run → Restart & clear) and re-run from the
top. Pip cannot swap out an already-imported module, so the restart is required rather than optional.

**Pinned, not latest.** `day1_config.json` froze the package set the 255-graph corpus run used:
circuit-tracer 0.5.0 with nnsight 0.7.0. A newer circuit-tracer expects
`nnsight.intervention.tracing`, which older nnsight builds lack — and more importantly, this
notebook's numbers are meant to be comparable with that run, so it should use that environment rather
than whatever is current.

The `diffusers` uninstall is from the project's own notes: on Python 3.13 a pinned `huggingface_hub`
leaves `diffusers` importing a symbol it no longer exports, and `nnsight` pulls `diffusers` in
transitively, so model loading fails with an `ImportError` that looks unrelated to anything here.

In [1]:
import glob, json
import importlib.metadata as md


def ver(p):
    try:
        return md.version(p)
    except Exception:
        return "absent"


WATCH = ("circuit-tracer", "nnsight", "torch", "transformers", "diffusers")
print("before:", {p: ver(p) for p in WATCH})

# what the corpus run actually used, per the frozen Day 1 config
hits = (glob.glob("/kaggle/input/**/day1_config.json", recursive=True)
        + glob.glob("/kaggle/working/**/day1_config.json", recursive=True)
        + glob.glob("./**/day1_config.json", recursive=True))
if hits:
    d1 = json.load(open(hits[0]))
    found = {k: v for k, v in d1.items()
             if any(t in k.lower() for t in ("packag", "version", "env"))}
    print(f"\nday1_config.json: {found if found else 'no version keys found'}")
else:
    print("\nday1_config.json not found - using the versions from the write-up")

print("\npinning circuit-tracer==0.5.0 and nnsight==0.7.0 (the corpus-run pair)")

before: {'circuit-tracer': 'absent', 'nnsight': 'absent', 'torch': '2.11.0+cu128', 'transformers': '5.16.1', 'diffusers': '0.40.0'}

day1_config.json not found - using the versions from the write-up

pinning circuit-tracer==0.5.0 and nnsight==0.7.0 (the corpus-run pair)


In [2]:
!pip install -q circuit-tracer==0.5.0 nnsight==0.7.0
!pip uninstall -y -q diffusers 2>/dev/null || true

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.2/153.2 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 272.5/272.5 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 968.6/968.6 kB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 96.0 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 71.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.5/82.5 kB 4.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the followi

In [3]:
import sys
import importlib.metadata as md


def ver(p):
    try:
        return md.version(p)
    except Exception:
        return "absent"


print("after:", {p: ver(p) for p in
                 ("circuit-tracer", "nnsight", "torch", "transformers", "diffusers")})

for m in [k for k in list(sys.modules)
          if k.split(".")[0] in ("circuit_tracer", "nnsight", "diffusers")]:
    sys.modules.pop(m, None)

try:
    import circuit_tracer
    from circuit_tracer import ReplacementModel, attribute
    print("\ncircuit_tracer imports cleanly - continue to Part 1")
except Exception as ex:
    print(f"\nimport failed: {type(ex).__name__}: {str(ex)[:300]}")
    print("\n-> RESTART THE KERNEL (Run > Restart & clear), then re-run from the top.")
    print("   Expected on the first pass: pip changed packages that were already")
    print("   imported, and only a restart picks them up.")
    print("   If it still fails after a restart, print the installed nnsight version and")
    print("   find the circuit-tracer release that matches it.")

after: {'circuit-tracer': '0.5.0', 'nnsight': '0.7.0', 'torch': '2.11.0+cu128', 'transformers': '4.57.3', 'diffusers': 'absent'}

circuit_tracer imports cleanly - continue to Part 1


## Part 1 — Pick an arm

**Verified transcoder sizes**, read from the HuggingFace file listings rather than estimated:

| Arm | Model | Transcoder weights | VRAM estimate | Hardware |
|---|---|---|---|---|
| `gemma-plt` | Gemma-2-2B | 7.4 GB | 12.5 GB *measured* | free T4 — **done, 255 graphs** |
| `llama-plt` | Llama-3.2-1B | **17.3 GB** (16 × 1.08 GB) | ~20 GB | 24 GB card |
| `llama-clt` | Llama-3.2-1B | **20.4 GB** (136 decoders) | ~23 GB | 24 GB card |
| `gemma-clt` | Gemma-2-2B | **28.5 GB** (351 decoders) | ~34 GB | A100 40GB |
| `qwen-plt` | Qwen-3 0.6B | 18.8 GB (28 × 671 MB) | ~20 GB | 24 GB card |

**A smaller model does not give a smaller transcoder.** Size goes as features × d_model × layers, and
these community sets all use large feature counts — Qwen-3 0.6B has 28 layers, so its per-layer set is
*larger* than Llama-1B's. GemmaScope at 7.4 GB is the outlier, which is why the original corpus run fit
on a free T4 and nothing else here does.

**So: one rented card, not two stages.** `llama-plt` and `llama-clt` are close enough in size that a
single 24 GB card (4090 / A10G / L4, roughly $0.40/hr) runs both in one session, loaded one at a time.
That is the PLT→CLT contrast with no model confound, in about an hour.

Disk is never the constraint: Kaggle's `/kaggle/temp` has ~1.1 TB free and the gate picks the roomiest
mount automatically.

**On the VRAM estimates.** They are weights + model, an upper bound. circuit-tracer offloads and
`lazy_encoder=True` keeps encoders off-device, so the real requirement is lower by an unknown margin.
`FORCE = True` downloads and attempts the load anyway; Part 4 then reports the measured peak, which is
worth more than my arithmetic.

In [14]:
ARMS = {
    "gemma-clt":    dict(model="google/gemma-2-2b",
                         transcoders="mntss/clt-gemma-2-2b-426k",
                         kind="CLT", n_layers=26, model_gb=9.8, vram_model_gb=5.2),
    "gemma-plt-tl": dict(model="google/gemma-2-2b",
                         transcoders="mntss/gemma-scope-transcoders",
                         kind="PLT", n_layers=26, model_gb=9.8, vram_model_gb=5.2),
    "llama-clt":    dict(model="meta-llama/Llama-3.2-1B",
                         transcoders="mntss/clt-llama-3.2-1b-524k",
                         kind="CLT", n_layers=16, model_gb=2.5, vram_model_gb=2.5),
    "llama-plt-tl": dict(model="meta-llama/Llama-3.2-1B",
                         transcoders="mntss/transcoder-Llama-3.2-1B",
                         kind="PLT", n_layers=16, model_gb=2.5, vram_model_gb=2.5),
}

# Run these in order, restarting the kernel between each (see the checklist below).
#   1. gemma-clt      the key arm: C1 on the model the 27/27 baseline came from
#   2. gemma-plt-tl   same-backend Gemma PLT baseline, 54 prompts
#   3. llama-clt      second model
#   4. llama-plt-tl   same-backend Llama PLT baseline
ARM = "llama-clt"

# The nnsight backend raises MissedProviderError on cross-layer transcoders: a CLT decoder
# writes into several downstream layers and the backend's experimental hook injection misses
# a module. TransformerLens works and also streams the decoders - the 20.4 GB Llama CLT
# peaked at 7.0 GB, not the ~23 GB estimated. Every arm here uses the same backend so the
# PLT->CLT contrast has no library difference inside it.
BACKEND = "transformerlens"

FORCE = True       # VRAM estimates assume resident weights, which is false for this
                   # backend. They over-state by 3x or more; ignore them and measure.

CFG_ARM = ARMS[ARM]
print(f"arm {ARM}: {CFG_ARM['model']} + {CFG_ARM['transcoders']}")
print(f"  {CFG_ARM['kind']}, {CFG_ARM['n_layers']} layers, backend={BACKEND}, FORCE={FORCE}")

arm llama-clt: meta-llama/Llama-3.2-1B + mntss/clt-llama-3.2-1b-524k
  CLT, 16 layers, backend=transformerlens, FORCE=True


### Per-arm checklist

Each arm needs its own session state. **Restart the kernel between arms** — it is the only
reliable way to free a loaded replacement model, and a stale `model` is how you would silently
attribute one arm's prompts with another arm's transcoders.

After a restart, run in this order:

| Cell | Run it? | Why |
|---|---|---|
| Part 0 install (3 cells) | **skip** after the first time | pip packages survive a kernel restart |
| Part 1 config | **yes** — change `ARM` | everything downstream reads `CFG_ARM` |
| Part 1b auth | yes | cheap, and `login()` state is lost on restart |
| Part 2 gate | yes | defines `HOME`, `vram`, `gpu`, and the arm's size figures |
| Part 3 inputs | yes | `MCFG["n_layers"]` is arm-specific — Gemma 26, Llama 16 |
| Part 3 task check | yes | writes `task_check_<arm>.csv`, which Part 7's secondary pass reads |
| Part 4 download + load | yes | cached repos re-resolve in seconds |
| Part 5 layout | **yes, never skip** | the assertion that makes the arm's numbers mean anything |
| Part 6 run | yes | appends to `frames_<arm>.jsonl` |
| Part 7 analysis | after any arm | reads every arm present |
| Part 8 save | after any arm | rebuilds the summary from all arms |

**Nothing is overwritten.** `frames_<arm>.jsonl` and `task_check_<arm>.csv` are arm-scoped. The
summary files are rewritten each time, but Part 7 regenerates them from every `frames_*.jsonl` on
disk, so they get more complete rather than losing earlier arms.

**Results now go to `/kaggle/working/out`**, which persists as notebook output. Only the HuggingFace
cache lives on `/kaggle/temp`, which is wiped when the session ends — the weights are
re-downloadable, your results are not.

One wrinkle worth knowing: `gemma-plt-tl` re-runs the Gemma PLT arm on this backend because the
255-graph corpus run used nnsight. Comparing it against those existing rows is a free
backend-equivalence check. If the two disagree by more than the 3.4×10⁻⁴ re-attribution ceiling,
that is a finding, and it needs settling before either Gemma number is trusted.

## Part 1b — HuggingFace authentication

In [15]:
# Llama-3.2-1B is GATED: accept the licence on its model page with the same account that
# owns this token, or the tokenizer load in Part 3 fails before anything else runs.
import os

_tok = os.environ.get("HF_TOKEN")
if not _tok:
    try:
        from kaggle_secrets import UserSecretsClient
        _tok = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as ex:
        print(f"no Kaggle secret ({type(ex).__name__}); set HF_TOKEN in the environment")
if _tok:
    os.environ["HF_TOKEN"] = os.environ["HUGGING_FACE_HUB_TOKEN"] = _tok
    from huggingface_hub import login, whoami
    login(token=_tok)
    try:
        print(f"authenticated as {whoami()['name']}")
    except Exception:
        print("token set")
else:
    print("NO TOKEN. Public repos work; gated ones (Llama) will not.")

if CFG_ARM["model"].startswith("meta-llama"):
    print(f"\n{CFG_ARM['model']} is gated. On 401/403, accept the licence at")
    print(f"  https://huggingface.co/{CFG_ARM['model']}")
    print("with the account owning this token, then re-run.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


authenticated as JSON9776

meta-llama/Llama-3.2-1B is gated. On 401/403, accept the licence at
  https://huggingface.co/meta-llama/Llama-3.2-1B
with the account owning this token, then re-run.


## Part 2 — Feasibility gate (metadata only, no downloads)

In [16]:
import sys, shutil, json, glob, gc, time
import numpy as np, pandas as pd
from scipy import stats

try:
    import torch
    has_cuda = torch.cuda.is_available()
    vram = (torch.cuda.get_device_properties(0).total_memory / 1e9) if has_cuda else 0.0
    gpu = torch.cuda.get_device_name(0) if has_cuda else "none"
except Exception as ex:
    has_cuda, vram, gpu = False, 0.0, f"torch unavailable: {ex}"


def free_gb(path):
    try:
        os.makedirs(path, exist_ok=True)
        return shutil.disk_usage(path).free / 1e9
    except Exception:
        return float("nan")


try:
    import psutil
    RAM_TOTAL = psutil.virtual_memory().total / 1e9
    RAM_FREE = psutil.virtual_memory().available / 1e9
except Exception:
    RAM_TOTAL = RAM_FREE = float("nan")

cands = {p: free_gb(p) for p in ("/kaggle/temp", "/kaggle/working", "/tmp",
                                 os.path.expanduser("~"))}
cands = {p: g for p, g in cands.items() if g == g}
HOME = max(cands, key=lambda p: cands[p])
print(f"GPU {gpu} | VRAM {vram:.1f} GB | RAM {RAM_TOTAL:.1f} GB "
      f"({RAM_FREE:.1f} free)")
for p, g in cands.items():
    print(f"  disk {p:18s} {g:7.1f} GB free" + ("   <- using this" if p == HOME else ""))

# File naming differs between repos (W_enc_N / W_dec_N for CLTs, layer_N for per-layer
# sets), so size by extension rather than by name. Anything under features/ is
# visualisation data that attribution never reads.
from huggingface_hub import HfApi
sib = HfApi().repo_info(CFG_ARM["transcoders"], files_metadata=True).siblings
need = [s for s in sib if not s.rfilename.startswith("features/")]
skip = [s for s in sib if s.rfilename.startswith("features/")]
W_GB = sum((s.size or 0) for s in need) / 1e9
S_GB = sum((s.size or 0) for s in skip) / 1e9
weights = [s for s in need if s.rfilename.endswith((".safetensors", ".pt", ".bin"))]
print(f"\n{CFG_ARM['transcoders']}")
print(f"  {W_GB:5.1f} GB to download ({len(weights)} weight files, "
      f"{len(need) - len(weights)} small)")
print(f"  {S_GB:5.1f} GB features/ skipped")
if weights:
    big = sorted(weights, key=lambda s: -(s.size or 0))[:3]
    print("  largest: " + ", ".join(f"{s.rfilename} {(s.size or 0)/1e9:.2f} GB" for s in big))

need_disk = W_GB + CFG_ARM["model_gb"]
est_vram = W_GB + CFG_ARM["vram_model_gb"]
# RAM is the binding constraint on the transformerlens backend: cross-layer decoders are
# held in CPU RAM and streamed to the device, so the whole transcoder set plus the bf16
# model must fit in RAM. This is what killed gemma-clt (33.7 GB needed, 31.7 available)
# while llama-clt ran fine at 22.9 GB.
need_ram = W_GB + CFG_ARM["vram_model_gb"]
OK_DISK = cands[HOME] > need_disk * 1.05
OK_RAM = RAM_FREE > need_ram * 1.02 if RAM_FREE == RAM_FREE else True
VRAM_LIKELY = vram > est_vram

print(f"\ndisk needed ~{need_disk:.1f} GB -> {cands[HOME]:.1f} GB free  "
      f"[{'OK' if OK_DISK else 'INSUFFICIENT'}]")
print(f"RAM needed ~{need_ram:.1f} GB -> {RAM_FREE:.1f} GB available  "
      f"[{'OK' if OK_RAM else 'TOO SMALL - this is the one that kills the kernel'}]")
print(f"VRAM estimate ~{est_vram:.1f} GB -> {vram:.1f} GB present  "
      f"[{'likely OK' if VRAM_LIKELY else 'probably fine anyway, see below'}]")
print("  VRAM is NOT the constraint on this backend - the 20.4 GB llama-clt peaked at")
print("  7.0 GB of VRAM because decoders stream from CPU RAM. Watch the RAM line.")

if not OK_RAM:
    print("\n" + "=" * 78)
    print(f"RAM is short by ~{need_ram - RAM_FREE:.1f} GB. Kaggle kills the kernel outright")
    print("on a RAM overcommit, so no exception handler will catch it.")
    print("=" * 78)
    print("Reclaim what you can: restart the kernel and skip the Part 3 task check, which")
    print("instantiates the HF model on CPU and may not release it. If still short, this")
    print("arm is not feasible here - lazy_decoder is already the default and does not help.")

if not OK_DISK:
    raise SystemExit(f"not enough disk for {ARM}: need ~{need_disk:.0f} GB on {HOME}")

if not VRAM_LIKELY and not FORCE:
    print("\n" + "=" * 78)
    print(f"Estimate says arm '{ARM}' probably will not fit. Nothing downloaded.")
    print("=" * 78)
    print(f"  1. FORCE = True to try anyway: a {W_GB:.0f} GB download and a few minutes buys")
    print("     the real number instead of this estimate.")
    print("  2. rent a 24 GB card (4090 / A10G / L4) and run both llama arms in one session.")
    print("\nverified weights: gemma-plt 7.4 | llama-plt 17.3 | llama-clt 20.4 | "
          "gemma-clt 28.5 GB")
    raise SystemExit(f"estimate says {ARM} does not fit; set FORCE=True to try regardless")

if not VRAM_LIKELY:
    print("\nFORCE is on: proceeding despite the estimate. Watch Part 4 for the real peak.")
print("\nproceeding")

GPU Tesla T4 | VRAM 15.6 GB | RAM 33.7 GB (28.8 free)
  disk /kaggle/temp        1125.9 GB free   <- using this
  disk /kaggle/working       20.9 GB free
  disk /tmp                1125.9 GB free
  disk /root               1125.9 GB free

mntss/clt-llama-3.2-1b-524k
   20.4 GB to download (32 weight files, 2 small)
    5.0 GB features/ skipped
  largest: W_dec_0.safetensors 2.15 GB, W_dec_1.safetensors 2.01 GB, W_dec_2.safetensors 1.88 GB

disk needed ~22.9 GB -> 1125.9 GB free  [OK]
RAM needed ~22.9 GB -> 28.8 GB available  [OK]
VRAM estimate ~22.9 GB -> 15.6 GB present  [probably fine anyway, see below]
  VRAM is NOT the constraint on this backend - the 20.4 GB llama-clt peaked at
  7.0 GB of VRAM because decoders stream from CPU RAM. Watch the RAM line.

FORCE is on: proceeding despite the estimate. Watch Part 4 for the real peak.

proceeding


## Part 3 — Inputs, the matched frames, and the task check

The prompts come from the existing corpus unchanged. The task check asks whether *this* model produces
the expected answer, because the corpus was validated on Gemma-2-2B.

In [17]:
os.environ["HF_HOME"] = f"{HOME}/hf"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# results persist in /kaggle/working; only the weight cache goes on the big ephemeral mount
OUT = "/kaggle/working/out" if os.path.isdir("/kaggle/working") else f"{HOME}/out"
os.makedirs(OUT, exist_ok=True)
print(f"results -> {OUT}   weights cache -> {HOME}/hf")


def find_input(name, required=True):
    hits = (glob.glob(f"/kaggle/working/**/{name}", recursive=True)
            + glob.glob(f"/kaggle/input/**/{name}", recursive=True)
            + glob.glob(f"./**/{name}", recursive=True))
    if not hits and required:
        raise FileNotFoundError(f"{name} - upload corpus.csv, results.csv, day3_config.json "
                                f"and ct_utils.py alongside this notebook")
    return hits[0] if hits else None


def free():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


src = find_input("ct_utils.py")
sys.path.insert(0, os.path.dirname(os.path.abspath(src)))
import ct_utils, importlib; importlib.reload(ct_utils)
from ct_utils import to_np, graph_metrics_fast

corpus = pd.read_csv(find_input("corpus.csv"),
                     dtype={"expected": str, "subtype": str, "prompt": str},
                     keep_default_na=False)
D3 = json.load(open(find_input("day3_config.json")))
AK = dict(D3["attr_kwargs"])
# the metric pipeline reads n_layers from config; this arm may differ from Gemma's 26
MCFG = {**D3, "n_layers": CFG_ARM["n_layers"]}
print(f"corpus {len(corpus)} | cap {AK['max_feature_nodes']} | "
      f"n_layers {MCFG['n_layers']} for this arm")

fr = corpus[corpus.category.isin(["factual_recall", "multi_hop"])].copy()
fr["frame"] = np.where(fr.subtype.str.contains("long"), "long",
               np.where(fr.subtype.str.contains("med"), "med", "short"))
frames = fr.groupby("expected").frame.agg(set)
PAIR_FACTS = sorted([k for k, v in frames.items() if {"short", "long"} <= v])
TRIPLE_FACTS = sorted([k for k in PAIR_FACTS if "med" in frames[k]])
todo = fr[fr.expected.isin(PAIR_FACTS)].copy()
print(f"{len(PAIR_FACTS)} facts with short+long ({len(TRIPLE_FACTS)} with medium), "
      f"{len(todo)} prompts")
assert len(PAIR_FACTS) >= 20, "too few matched facts - check the subtype encoding"

results -> /kaggle/working/out   weights cache -> /kaggle/temp/hf
corpus 255 | cap 32768 | n_layers 16 for this arm
27 facts with short+long (12 with medium), 66 prompts


In [25]:
# does THIS model answer these prompts? The corpus was validated on Gemma-2-2b.
from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained(CFG_ARM["model"])
hf = AutoModelForCausalLM.from_pretrained(CFG_ARM["model"], torch_dtype=torch.bfloat16,
                                          low_cpu_mem_usage=True).to("cuda").eval()
hits = []
with torch.no_grad():
    for _, r in todo.iterrows():
        ids = tok(r.prompt, return_tensors="pt").to("cuda")
        out = hf.generate(**ids, max_new_tokens=4, do_sample=False,
                          pad_token_id=tok.eos_token_id)
        gen = tok.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True)
        hits.append(dict(prompt_id=r.prompt_id, frame=r.frame, expected=r.expected,
                         generated=gen.strip(),
                         ok=bool(r.expected) and r.expected.strip().lower()
                            in gen.strip().lower()))
TASK = pd.DataFrame(hits)
rate = TASK.ok.mean()
print(f"\nanswer rate on {CFG_ARM['model']}: {rate:.0%} ({TASK.ok.sum()}/{len(TASK)})")
print(TASK.groupby("frame").ok.agg(["mean", "size"]).round(3).to_string())
print("\nGemma-2-2b: 89% on short frames with the Fact: prefix, 94% multi-hop.")
if rate < 0.5:
    print("\n! Under 50%. The graphs will mostly explain something other than the answer.")
    print("  The length test remains a valid test of the metric's response to padding, but")
    print("  say so rather than implying a like-for-like replication.")
print(TASK[~TASK.ok].head(8).to_string(index=False))
TASK.to_csv(f"{OUT}/task_check_{ARM}.csv", index=False)

del hf; free()


answer rate on meta-llama/Llama-3.2-1B: 71% (47/66)
        mean  size
frame             
long   0.852    27
med    0.000    12
short  0.889    27

Gemma-2-2b: 89% on short frames with the Fact: prefix, 94% multi-hop.
prompt_id frame expected                 generated    ok
    p0001   med    Paris   the most beautiful city False
    p0004   med    Tokyo the most populous country False
    p0007   med     Rome   the most beautiful city False
    p0010   med    Cairo      the capital of Egypt False
    p0013   med Brasilia       the largest city in False
    p0014  long Brasilia       the largest city in False
    p0016   med  Nairobi       the largest city in False
    p0019   med     Oslo      located in the north False


## Part 4 — Download transcoders and load the replacement model

In [18]:
from huggingface_hub import snapshot_download

t0 = time.time()
tc_path = snapshot_download(CFG_ARM["transcoders"],
                            allow_patterns=["*.safetensors", "*.json", "*.yaml", "*.pt"],
                            ignore_patterns=["features/*"])
got = sum(os.path.getsize(os.path.join(r, f))
          for r, _, fs in os.walk(tc_path) for f in fs) / 1e9
print(f"{got:.1f} GB in {(time.time()-t0)/60:.1f} min -> {tc_path}")
print(f"disk free now {free_gb(HOME):.1f} GB")

# free any model left over from a previous arm. Restarting the kernel between arms is the
# reliable way; this is a belt-and-braces guard against attributing with the wrong weights.
for _v in ("model", "g_ref", "hf"):
    if _v in globals():
        del globals()[_v]
free()

from circuit_tracer import ReplacementModel, attribute

torch.cuda.reset_peak_memory_stats()
t0 = time.time()
try:
    model = ReplacementModel.from_pretrained(CFG_ARM["model"], tc_path, backend=BACKEND,
                                             dtype=torch.bfloat16, device="cuda",
                                             lazy_encoder=True)
except torch.cuda.OutOfMemoryError as ex:
    print("\n" + "!" * 78)
    print(f"OUT OF MEMORY loading '{ARM}' on {gpu} ({vram:.1f} GB)")
    print("!" * 78)
    print("try lazy_decoder=True, or drop lazy_encoder. Batch size will not help if the")
    print("weights themselves do not fit.")
    print(f"\n{str(ex)[:300]}")
    raise
peak = torch.cuda.max_memory_allocated() / 1e9
print(f"loaded in {(time.time()-t0)/60:.1f} min, backend={BACKEND}")
print(f"VRAM allocated {torch.cuda.memory_allocated()/1e9:.1f} GB, peak {peak:.1f} GB, "
      f"of {vram:.1f} GB   (pre-run estimate was {est_vram:.1f} GB)")

Fetching 33 files:   0%|          | 0/33 [00:00<?, ?it/s]

W_dec_12.safetensors:   0%|          | 0.00/537M [00:00<?, ?B/s]

W_dec_10.safetensors:   0%|          | 0.00/805M [00:00<?, ?B/s]

W_dec_1.safetensors:   0%|          | 0.00/2.01G [00:00<?, ?B/s]

W_dec_11.safetensors:   0%|          | 0.00/671M [00:00<?, ?B/s]

...

W_enc_7.safetensors:   0%|          | 0.00/134M [00:00<?, ?B/s]

W_enc_8.safetensors:   0%|          | 0.00/134M [00:00<?, ?B/s]

W_enc_9.safetensors:   0%|          | 0.00/134M [00:00<?, ?B/s]

config.yaml:   0%|          | 0.00/148 [00:00<?, ?B/s]

20.4 GB in 1.6 min -> /root/.cache/huggingface/hub/models--mntss--clt-llama-3.2-1b-524k/snapshots/d6ab3ffa8184c9894b2ffb35093c50c098c0f223
disk free now 1105.5 GB
Loaded pretrained model meta-llama/Llama-3.2-1B into HookedTransformer
loaded in 0.4 min, backend=transformerlens
VRAM allocated 3.0 GB, peak 3.4 GB, of 15.6 GB   (pre-run estimate was 22.9 GB)


## Part 5 — Re-validate the node layout

**The cell most likely to fail, and it must fail loudly.** The metric pipeline assumes
`[features | errors (layer-major) | tokens | logits]` with `n_features = len(selected_features)`.
A different model changes `n_layers`; a cross-layer transcoder may change the feature organisation
entirely, since it has one encoder per layer but decoders spanning layers.

Nothing below is trustworthy until `n_feat + n_layers*n_tok + n_tok + n_logits == n_nodes` holds on a
real graph from this arm. If it does not, re-derive the layout from the graph — as Day 1 did for Gemma
PLTs, and for the same reason: locating nodes by assumption rather than by structure already cost this
project a day and one wrong finding.

In [19]:
probe = todo.iloc[0]
g_ref = attribute(prompt=probe.prompt, model=model, **AK)

N = g_ref.adjacency_matrix.shape[0]
nt = int(len(to_np(g_ref.input_tokens)))
nlg = int(len(to_np(g_ref.logit_probabilities)))
n_feat = int(len(g_ref.selected_features))
NLa = MCFG["n_layers"]
expected_N = n_feat + NLa * nt + nt + nlg
print(f"probe {probe.prompt_id!r}: nodes {N} | tokens {nt} | logits {nlg} | features {n_feat}")
print(f"n_feat + {NLa}*{nt} + {nt} + {nlg} = {expected_N}   vs n_nodes = {N}")

LAYOUT_OK = (expected_N == N)
if not LAYOUT_OK:
    print("\n" + "!" * 78)
    print("LAYOUT MISMATCH - the Gemma PLT convention does not hold for this arm")
    print("!" * 78)
    print(f"off by {N - expected_N} nodes")
    A = g_ref.adjacency_matrix
    zin = (A.abs().sum(1) == 0).cpu().numpy()
    runs, s = [], None
    for i, z in enumerate(zin):
        if z and s is None: s = i
        if not z and s is not None: runs.append((s, i - 1)); s = None
    if s is not None: runs.append((s, len(zin) - 1))
    print(f"zero-in-degree runs: {runs[:10]}")
    print(f"zero-out-degree nodes: {int((A.abs().sum(0) == 0).sum())}")
    print("\nErrors and embeddings form one contiguous zero-in-degree run, errors first.")
    print("Confirm the error block's major axis from the mod-n_tok zero pattern. Logits are")
    print("the last n_logits nodes in logit_probabilities order - do NOT locate them by")
    print("zero out-degree, which also catches dead features and mis-orders probabilities.")
else:
    print("layout matches the convention")

m_ref = graph_metrics_fast(g_ref, MCFG)
print(f"\nreference replacement {m_ref['replacement_score']:.4f}  "
      f"mean path length {m_ref.get('mean_path_length', float('nan')):.3f}")
print("Gemma PLT corpus: replacement ~0.72, path length ~3.7")
assert LAYOUT_OK, "stop: layout mismatch, see above"

<sys>:0: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.


probe 'p0000': nodes 1369 | tokens 8 | logits 10 | features 1223
n_feat + 16*8 + 8 + 10 = 1369   vs n_nodes = 1369
layout matches the convention

reference replacement 0.6563  mean path length 2.826
Gemma PLT corpus: replacement ~0.72, path length ~3.7


## Part 6 — Run the matched frames for this arm

In [20]:
ARM_OUT = f"{OUT}/frames_{ARM}.jsonl"
done = ({json.loads(l)["prompt_id"] for l in open(ARM_OUT) if l.strip()}
        if os.path.exists(ARM_OUT) else set())
run = todo[~todo.prompt_id.isin(done)]
print(f"{len(done)} done, {len(run)} to go")

t0, times = time.time(), []
for i, (_, r) in enumerate(run.iterrows(), 1):
    t1 = time.time()
    try:
        g = attribute(prompt=r.prompt, model=model, **AK)
        rec = dict(prompt_id=r.prompt_id, category=r.category, subtype=r.subtype,
                   expected=r.expected, arm=ARM, kind=CFG_ARM["kind"],
                   model=CFG_ARM["model"], transcoders=CFG_ARM["transcoders"],
                   backend=BACKEND, status="ok",
                   **{k: v for k, v in graph_metrics_fast(g, MCFG).items()
                      if not isinstance(v, (list, dict))})
        rec.update(n_features=int(len(g.selected_features)),
                   n_nodes=int(g.adjacency_matrix.shape[0]),
                   n_tok=int(len(to_np(g.input_tokens))))
        del g
    except Exception as ex:
        rec = dict(prompt_id=r.prompt_id, category=r.category, subtype=r.subtype,
                   expected=r.expected, arm=ARM, backend=BACKEND, status="error",
                   error=f"{type(ex).__name__}: {str(ex)[:200]}")
    free()
    times.append(time.time() - t1)
    with open(ARM_OUT, "a") as f:
        f.write(json.dumps(rec, default=float) + "\n"); f.flush(); os.fsync(f.fileno())
    if i % 5 == 0 or rec["status"] != "ok":
        print(f"{i:3d}/{len(run)}  {r.prompt_id}  {times[-1]:5.1f}s  {rec['status']}"
              + ("" if rec["status"] == "ok" else f"  {rec.get('error','')}"))

print(f"\n{len(times)} attributed, mean {np.mean(times) if times else float('nan'):.1f}s, "
      f"{(time.time()-t0)/60:.1f} min total")
print("\n>>> to add another arm: change ARM in Part 1, restart the kernel, re-run. "
      "Results accumulate in separate files. <<<")

0 done, 66 to go
  5/66  p0004    3.0s  ok
 10/66  p0009    2.3s  ok
 15/66  p0014    5.1s  ok
 20/66  p0019    3.1s  ok
 25/66  p0024    2.4s  ok
 30/66  p0029    5.1s  ok
 35/66  p0034    3.1s  ok
 40/66  p0097    4.3s  ok
 45/66  p0102    3.2s  ok
 50/66  p0107    4.1s  ok
 55/66  p0112    3.4s  ok
 60/66  p0117    2.3s  ok
 65/66  p0124    4.3s  ok

66 attributed, mean 3.6s, 4.0 min total

>>> to add another arm: change ARM in Part 1, restart the kernel, re-run. Results accumulate in separate files. <<<


## Part 7 — The length effect, per arm and across arms

Reads every `frames_*.jsonl` present, plus the Gemma PLT rows from `results.csv`, and tests each arm
against the same threshold.

**Two passes, following D3.9.** Primary is all matched pairs. Secondary restricts to facts where the
model produced the expected answer in *both* frames, read from `task_check_<arm>.csv`. The primary is
the headline because an attribution graph is a valid object whatever the model predicted — the metric's
response to padding is the metric's response to padding. The secondary matters because if the two
diverge, the length effect is partly about prompts where the model did something other than the task.

**Medium frames are excluded throughout.** On Llama-3.2-1B they elicited a descriptive continuation
rather than the answer (0 of 12), so the three-frame Friedman trend has no counterpart here and is not
attempted. That is reported as a limitation of the replication, not worked around.

In [26]:
import glob, shutil, os
src = glob.glob(f"{OUT}/task_check_llama-*.csv")
print("have:", [os.path.basename(s) for s in src])
for target in ("llama-clt", "llama-plt-tl"):
    dst = f"{OUT}/task_check_{target}.csv"
    if src and not os.path.exists(dst):
        shutil.copy(src[0], dst); print(f"wrote {os.path.basename(dst)}")
print("now:", [os.path.basename(s) for s in glob.glob(f"{OUT}/task_check_*.csv")])

have: ['task_check_llama-clt.csv']
wrote task_check_llama-plt-tl.csv
now: ['task_check_llama-clt.csv', 'task_check_llama-plt-tl.csv']


In [27]:
def logit_(p, eps=1e-6):
    p = np.clip(np.asarray(p, float), eps, 1 - eps)
    return np.log(p / (1 - p))


def ols1(x, y):
    x = np.asarray(x, float); y = np.asarray(y, float)
    X = np.column_stack([np.ones(len(y)), x])
    if np.linalg.matrix_rank(X) < 2 or len(y) < 3:
        return np.nan, np.nan, np.nan, np.nan
    b, *_ = np.linalg.lstsq(X, y, rcond=None)
    r = y - X @ b
    dof = len(y) - 2
    cov = ((r ** 2).sum() / dof) * np.linalg.pinv(X.T @ X)
    se = float(np.sqrt(max(cov[1, 1], 0)))
    r2 = 1 - (r ** 2).sum() / max(((y - y.mean()) ** 2).sum(), 1e-30)
    p = float(2 * stats.t.sf(abs(b[1] / se), dof)) if se > 0 else np.nan
    return float(b[1]), se, p, float(r2)


def frame_of(s):
    s = s.astype(str)
    return np.where(s.str.contains("long"), "long",
            np.where(s.str.contains("med"), "med", "short"))


def build_pairs(d, keep_facts=None):
    """Short/long pairs per fact. keep_facts restricts to a set of `expected` values."""
    d = d.copy()
    d["frame"] = frame_of(d.subtype)
    d = d[d.frame.isin(["short", "long"])]          # medium is never used
    if keep_facts is not None:
        d = d[d.expected.isin(keep_facts)]
    cols = [c for c in ("replacement_score", "n_tok", "n_features") if c in d.columns]
    piv = {c: d.pivot_table(index="expected", columns="frame", values=c, aggfunc="mean")
           for c in cols}
    if not all({"short", "long"} <= set(v.columns) for v in piv.values()):
        return None
    P = pd.DataFrame({f"{c}__{f}": piv[c][f] for c in piv
                      for f in ("short", "long")}).dropna()
    if len(P) < 3:
        return None
    P["d_lg"] = logit_(P.replacement_score__long) - logit_(P.replacement_score__short)
    P["d_raw"] = P.replacement_score__long - P.replacement_score__short
    P["d_tok"] = P.n_tok__long - P.n_tok__short
    P["d_feat"] = (P.n_features__long - P.n_features__short
                   if "n_features__long" in P.columns else np.nan)
    return P


def row_for(name, d, P, pass_name):
    w = stats.wilcoxon(P.d_lg) if len(P) >= 5 else None
    b, se, p, r2 = ols1(P.d_tok, P.d_lg)
    return dict(arm=name, pass_=pass_name,
                kind=d.kind.iloc[0] if "kind" in d.columns else "?",
                n_pairs=len(P), positive=int((P.d_lg > 0).sum()),
                mean_raw=float(P.d_raw.mean()),
                wilcoxon_p=float(w.pvalue) if w else np.nan,
                slope=b, slope_p=p, slope_r2=r2,
                mean_repl=float(d.replacement_score.mean()),
                mean_path=float(d.mean_path_length.mean())
                          if "mean_path_length" in d.columns else np.nan,
                r_dtok_dfeat=float(P.d_tok.corr(P.d_feat))
                             if P.d_feat.notna().any() else np.nan)


# ---- collect arms --------------------------------------------------------
arms = {}
for f in sorted(glob.glob(f"{OUT}/frames_*.jsonl")):
    d = pd.DataFrame([json.loads(l) for l in open(f) if l.strip()])
    d = d[d.status == "ok"]
    if len(d):
        arms[d.arm.iloc[0]] = d

gp = pd.read_csv(find_input("results.csv"), keep_default_na=False)
for c in ("replacement_score", "mean_path_length", "n_features", "n_tok"):
    if c in gp.columns:
        gp[c] = pd.to_numeric(gp[c], errors="coerce")
if "expected" not in gp.columns:
    gp = gp.merge(corpus[["prompt_id", "expected"]], on="prompt_id", how="left")
gp = gp[gp.prompt_id.isin(set(todo.prompt_id))].copy()
if len(gp):
    gp["arm"], gp["kind"] = "gemma-plt", "PLT"
    arms["gemma-plt"] = gp
print(f"arms available: {list(arms)}\n")

# ---- facts answered in BOTH frames, per arm -----------------------------
def answered_both(name):
    f = f"{OUT}/task_check_{name}.csv"
    if os.path.exists(f):
        t = pd.read_csv(f, keep_default_na=False)
        t = t[t.frame.isin(["short", "long"])]
        g = t.groupby("expected").ok.agg(["sum", "size"])
        return set(g[(g["sum"] == g["size"]) & (g["size"] >= 2)].index)
    # any Gemma arm can fall back to the corpus task_ok: the answer rate is a property
    # of the model and the prompt, not of the transcoder, so it transfers across arms on
    # the same model. This also lets the Gemma arms skip the task check to save RAM.
    if name.startswith("gemma") and "task_ok" in corpus.columns:
        t = corpus[corpus.prompt_id.isin(set(todo.prompt_id))].copy()
        t["frame"] = frame_of(t.subtype)
        t = t[t.frame.isin(["short", "long"])]
        t["ok"] = t.task_ok.astype(str).str.lower().isin(["true", "1", "1.0"])
        g = t.groupby("expected").ok.agg(["sum", "size"])
        return set(g[(g["sum"] == g["size"]) & (g["size"] >= 2)].index)
    return None


rows, pairs = [], {}
for name, d in arms.items():
    P = build_pairs(d)
    if P is None:
        print(f"{name}: no short/long pairs, skipped"); continue
    rows.append(row_for(name, d, P, "all"))
    pairs[name] = P
    keep = answered_both(name)
    if keep is not None:
        Q = build_pairs(d, keep_facts=keep)
        if Q is not None:
            rows.append(row_for(name, d, Q, "answered"))
            pairs[f"{name}__answered"] = Q
            print(f"{name}: {len(P)} pairs all, {len(Q)} with both frames answered")
        else:
            print(f"{name}: {len(P)} pairs all; too few answered-both pairs to test")
    else:
        print(f"{name}: {len(P)} pairs all; no task check available for the secondary pass")

R = pd.DataFrame(rows)
print()
print(R.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

print("\n" + "=" * 82)
for _, r in R.iterrows():
    thr = min(24, int(np.ceil(0.88 * r.n_pairs)))
    verdict = "HOLDS" if r.positive >= thr else "FAILS"
    print(f"{r.arm:11s} [{r.pass_:8s}] {int(r.positive):2d}/{int(r.n_pairs):2d} positive "
          f"(need {thr})  slope {r.slope:+.5f} logit/token (p={r.slope_p:.3g})  -> {verdict}")
print("=" * 82)
print("Gemma PLT reference, 255-graph run: 27/27, +0.0458 logit/token, R2 0.81")
print("Primary pass is 'all'. If 'answered' disagrees, say so - it means the effect")
print("depends on prompts where the model did something other than the task.")

A = R[R.pass_ == "all"].set_index("arm")
if {"llama-plt", "llama-clt"} <= set(A.index):
    print(f"\nC2 slope (same model) {A.loc['llama-plt','slope']:+.5f} -> "
          f"{A.loc['llama-clt','slope']:+.5f}   "
          f"ratio {A.loc['llama-clt','slope'] / A.loc['llama-plt','slope']:.2f}  -> "
          f"{'HOLDS' if A.loc['llama-clt','slope'] < A.loc['llama-plt','slope'] else 'FAILS'}")
    print(f"C3 replacement {A.loc['llama-plt','mean_repl']:.4f} -> "
          f"{A.loc['llama-clt','mean_repl']:.4f}  -> "
          f"{'HOLDS' if A.loc['llama-clt','mean_repl'] > A.loc['llama-plt','mean_repl'] else 'FAILS'}")
    print(f"C4 path length {A.loc['llama-plt','mean_path']:.3f} -> "
          f"{A.loc['llama-clt','mean_path']:.3f}  -> "
          f"{'HOLDS' if A.loc['llama-clt','mean_path'] < A.loc['llama-plt','mean_path'] else 'FAILS'}")
else:
    print("\nC2-C4 need both llama-plt and llama-clt. Run the other arm.")

arms available: ['llama-clt', 'llama-plt-tl', 'gemma-plt']

llama-clt: 27 pairs all, 22 with both frames answered
llama-plt-tl: 27 pairs all, 22 with both frames answered
gemma-plt: 27 pairs all, 23 with both frames answered

         arm    pass_ kind  n_pairs  positive  mean_raw  wilcoxon_p  slope  slope_p  slope_r2  mean_repl  mean_path  r_dtok_dfeat
   llama-clt      all  CLT       27        27    0.0328      0.0000 0.0281   0.0001    0.4862     0.7133     2.7597        0.9998
   llama-clt answered  CLT       22        22    0.0339      0.0000 0.0282   0.0008    0.4372     0.7133     2.7597        0.9998
llama-plt-tl      all  PLT       27        21    0.0093      0.0000 0.0087   0.0472    0.1484     0.7366     3.4533        0.9996
llama-plt-tl answered  PLT       22        18    0.0105      0.0001 0.0077   0.1520    0.0998     0.7366     3.4533        0.9996
   gemma-plt      all  PLT       27        27    0.0323      0.0000 0.0457   0.0000    0.8073     0.7348     3.1520        0

In [12]:
import pandas as pd
rows = [dict(arm=n, d_tok=P.d_tok.mean(), d_feat=P.d_feat.mean(),
             dens=P.d_feat.mean()/P.d_tok.mean(),
             per_tok=P.d_raw.mean()/P.d_tok.mean(),
             per_feat=P.d_raw.mean()/P.d_feat.mean())
        for n, P in pairs.items() if "__answered" not in n]
A = pd.DataFrame(rows).sort_values("dens")
print(A.to_string(index=False, float_format=lambda v: f"{v:.6g}"))
print(f"\nper_feat spread {A.per_feat.max()/A.per_feat.min():.2f}x   "
      f"per_tok spread {A.per_tok.max()/A.per_tok.min():.2f}x")

         arm   d_tok  d_feat    dens    per_tok    per_feat
llama-plt-tl 4.77778 2601.26  544.45 0.00195538 3.59149e-06
   gemma-plt 4.77778 4442.48 929.822 0.00676906 7.27996e-06

per_feat spread 2.03x   per_tok spread 3.46x


In [13]:
import shutil
shutil.copy(f"{OUT}/task_check_llama-plt.csv", f"{OUT}/task_check_llama-plt-tl.csv")

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/out/task_check_llama-plt.csv'

## Part 8 — Save

In [23]:
summary = dict(question="does the length effect survive a cross-layer transcoder",
               arms_run=list(pairs), per_arm=R.to_dict("records"),
               gemma_plt_reference=dict(positive="27/27", slope=0.04575, r2=0.81,
                                        mean_raw=0.032, wilcoxon_p=1.5e-08),
               thresholds=dict(C0=24, C1=24, C2="CLT slope < PLT slope, same model"),
               gpu=gpu, vram_gb=round(vram, 1))
json.dump(summary, open(f"{OUT}/day8b_summary.json", "w"), indent=2, default=float)
R.to_csv(f"{OUT}/day8b_arms.csv", index=False)
for name, P in pairs.items():
    P.to_csv(f"{OUT}/pairs_{name}.csv")
print(json.dumps(summary["per_arm"], indent=2, default=float)[:1500])
print(f"\nwrote {OUT}/day8b_summary.json, day8b_arms.csv, pairs_*.csv")

[
  {
    "arm": "llama-clt",
    "pass_": "all",
    "kind": "CLT",
    "n_pairs": 27,
    "positive": 27,
    "mean_raw": 0.03281212614252627,
    "wilcoxon_p": 1.4901161193847656e-08,
    "slope": 0.028098943786606878,
    "slope_p": 5.29881535630575e-05,
    "slope_r2": 0.48622631122182347,
    "mean_repl": 0.7133146339442131,
    "mean_path": 2.7596747857543953,
    "r_dtok_dfeat": 0.9997542066955151
  },
  {
    "arm": "llama-plt-tl",
    "pass_": "all",
    "kind": "PLT",
    "n_pairs": 27,
    "positive": 21,
    "mean_raw": 0.00934238854811003,
    "wilcoxon_p": 3.0234456062316895e-05,
    "slope": 0.008703600072692082,
    "slope_p": 0.04718610750953071,
    "slope_r2": 0.14844195404516747,
    "mean_repl": 0.7366252877472474,
    "mean_path": 3.453280064354817,
    "r_dtok_dfeat": 0.9995994491360941
  },
  {
    "arm": "gemma-plt",
    "pass_": "all",
    "kind": "PLT",
    "n_pairs": 27,
    "positive": 27,
    "mean_raw": 0.032341071881707595,
    "wilcoxon_p": 1.490116119

In [24]:
import pandas as pd
rows = [dict(arm=n, d_tok=P.d_tok.mean(), d_feat=P.d_feat.mean(),
             dens=P.d_feat.mean()/P.d_tok.mean(),
             per_tok=P.d_raw.mean()/P.d_tok.mean(),
             per_feat=P.d_raw.mean()/P.d_feat.mean())
        for n, P in pairs.items() if "__answered" not in n]
A = pd.DataFrame(rows).sort_values("dens")
print(A.to_string(index=False, float_format=lambda v: f"{v:.6g}"))
print(f"\nper_feat spread {A.per_feat.max()/A.per_feat.min():.2f}x   "
      f"per_tok spread {A.per_tok.max()/A.per_tok.min():.2f}x")

         arm   d_tok  d_feat    dens    per_tok    per_feat
   llama-clt 4.77778 1446.78 302.814 0.00686765 2.26795e-05
llama-plt-tl 4.77778 2601.26  544.45 0.00195538 3.59149e-06
   gemma-plt 4.77778 4442.48 929.822 0.00676906 7.27996e-06

per_feat spread 6.31x   per_tok spread 3.51x


## Record in the pre-registration

> **Day 8 — cross-layer transcoder, length effect only [DATE].** Exploratory extension; C0–C4
> registered before any transcoder was loaded. Run on Llama-3.2-1B rather than Gemma-2-2B because the
> smallest Gemma CLT needs ~34 GB of VRAM: Llama has 16 layers, so 136 cross-layer decoder matrices
> instead of 351, and both a per-layer and a cross-layer transcoder exist for it, which removes the
> model confound from the PLT→CLT contrast.
>
> Arms: `llama-plt` on [GPU], `llama-clt` on [GPU]. Measured VRAM peak [__] GB against an estimate of
> [__]. Neither is free-tier — a departure from the project's free-compute principle, reported as a
> scope note. A smaller *model* does not help: transcoder size goes as features × d_model × layers,
> and the Qwen-3 0.6B per-layer set is 18.8 GB against Llama-1B's 17.3 GB.
>
> Task check: [model] answered [__]% of the matched-frame prompts, against 89–94% for Gemma-2-2B.
> [If low: the graphs explain the model's actual prediction rather than the answer, and this is
> reported as a test of the metric's response to padding rather than a like-for-like replication.]
>
> Node layout re-validated per arm: [matched / re-derived as __].
>
> C0 [__]/[__] pairs → [HOLDS/FAILS]. C1 [__]/[__] → [HOLDS/FAILS]. C2 slope [__] → [__]
> logit/token. C3 replacement [__] → [__]. C4 path length [__] → [__].
>
> No ratings were re-collected. P5 remains a Gemma per-layer result.

### What this settles and what it does not

**Settles:** whether the length effect is specific to per-layer transcoders, and whether it holds on a
second model. Two of the open questions the write-up lists.

**Does not settle:** whether the metric tracks human readability under a CLT. That needs the rating
protocol re-run on CLT graphs — a fresh sample, the viewer re-pointed at CLT features, feature labels
that may not exist for this dictionary, and a rater who has not already seen the per-layer versions.
P5 stays a per-layer result and should be labelled as one.

**Also does not settle the Gemma CLT question directly.** If the length effect behaves differently on
Llama, that could be the model rather than the dictionary. The `gemma-clt` arm is configured and ready
if an A100 becomes available.